In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [15]:
# =========================
# 1. CSV 로드
# =========================
csv_path = "/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/ace_safe_ver/splits/scaffold_by_endpoint_unseen_ver/split_summary.csv"
df_stats = pd.read_csv(csv_path)

# 숫자형 컬럼 보정
num_cols = ["n_total", "n_valid_smiles", "n_train", "n_valid", "n_test"]
for col in num_cols:
    df_stats[col] = pd.to_numeric(df_stats[col], errors="coerce").fillna(0).astype(int)

/var/folders/r2/hf806xw17vv9p1hn3v10qr980000gn/T/ipykernel_32842/1897195926.py:10: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_stats[col] = pd.to_numeric(df_stats[col], errors="coerce").fillna(0).astype(int)
/var/folders/r2/hf806xw17vv9

In [16]:
df_stats.head()

,dataset_name,endpoint,smiles_col,split_method,split_type,n_total,n_valid_smiles,n_train,n_valid,n_test,note
0,ames,ames,toxic_safe_decoded_smiles,scaffold,train_test,7490,7490,6780,0,710,"9:1 scaffold, min_test=30"
1,clintox,clintox,toxic_safe_decoded_smiles,scaffold,unseen_endpoint_test,54,54,0,0,54,n_total<=89; full test
2,dictrank,dictrank,toxic_safe_decoded_smiles,scaffold,unseen_endpoint_test,40,40,0,0,40,n_total<=89; full test
3,dilist,dilist,toxic_safe_decoded_smiles,scaffold,train_test,101,101,71,0,30,"9:1 scaffold, min_test=30"
4,diril,diril,toxic_safe_decoded_smiles,scaffold,unseen_endpoint_test,7,7,0,0,7,n_total<=89; full test


In [10]:
total_test_num = 0
for n_test_num in df_stats['n_total']:
    total_test_num += n_test_num
total_test_num

63251

In [11]:
total_test_num = 0
for n_test_num in df_stats['n_test']:
    total_test_num += n_test_num
total_test_num

6243

In [12]:
# =========================
# 2. dataset별 요약 통계
# =========================
dataset_summary = (
    df_stats.groupby("dataset_name", as_index=False)
    .agg(
        n_endpoints=("endpoint", "count"),
        total_samples=("n_total", "sum"),
        total_train=("n_train", "sum"),
        total_valid=("n_valid", "sum"),
        total_test=("n_test", "sum"),
    )
    .sort_values("total_samples", ascending=False)
)

print("=== Dataset별 요약 ===")
print(dataset_summary.to_string(index=False), "\n")

=== Dataset별 요약 ===
 dataset_name  n_endpoints  total_samples  total_train  total_valid  total_test
     tox21_df           12          37342        34527            0        2815
   metabolism            5          10032         9030            0        1002
         ames            1           7490         6780            0         710
 herg_unified            1           5520         4968            0         552
        sider           27           2528         1525            0        1003
skin_reaction            1            137          107            0          30
       dilist            1            101           71            0          30
      clintox            1             54            0            0          54
     dictrank            1             40            0            0          40
        diril            1              7            0            0           7 



In [17]:
# =========================
# 3. endpoint별 요약 통계
# =========================
endpoint_summary = (
    df_stats[["dataset_name", "endpoint", "n_total", "n_train", "n_valid", "n_test", "split_type"]]
    .sort_values(["n_total", "dataset_name"], ascending=[False, True])
)

print("=== Endpoint별 요약 (n_total 큰 순) ===")
print(endpoint_summary.to_string(index=False),"\n")

=== Endpoint별 요약 (n_total 큰 순) ===
 dataset_name                                                            endpoint  n_total  n_train  n_valid  n_test           split_type
         ames                                                                ames     7490     6780        0     710           train_test
     tox21_df                                                         tox21_NR-ER     7275     6808        0     467           train_test
     tox21_df                                                        tox21_NR-AhR     6051     5649        0     402           train_test
     tox21_df                                                        tox21_SR-MMP     6022     5450        0     572           train_test
 herg_unified                                                        herg_unified     5520     4968        0     552           train_test
     tox21_df                                                        tox21_SR-ARE     3867     3587        0     280           train_test

In [14]:
# =========================
# 4. 작은 endpoint 개수 확인
# =========================
bins_summary = {
    "n_total < 5": (df_stats["n_total"] < 5).sum(),
    "5 <= n_total < 10": ((df_stats["n_total"] >= 5) & (df_stats["n_total"] < 10)).sum(),
    "10 <= n_total < 30": ((df_stats["n_total"] >= 10) & (df_stats["n_total"] < 30)).sum(),
    "30 <= n_total < 100": ((df_stats["n_total"] >= 30) & (df_stats["n_total"] < 100)).sum(),
    "n_total >= 100": (df_stats["n_total"] >= 100).sum(),
}
print("=== Endpoint 크기 구간별 개수 ===")
for k, v in bins_summary.items():
    print(f"{k}: {v}")
print()

=== Endpoint 크기 구간별 개수 ===
n_total < 5: 1
5 <= n_total < 10: 1
10 <= n_total < 30: 1
30 <= n_total < 100: 13
n_total >= 100: 35



In [ ]:
# =========================
# 6. imbalance 확인용 비율 컬럼
# =========================
df_stats["train_ratio"] = df_stats["n_train"] / df_stats["n_total"].replace(0, 1)
df_stats["valid_ratio"] = df_stats["n_valid"] / df_stats["n_total"].replace(0, 1)
df_stats["test_ratio"]  = df_stats["n_test"]  / df_stats["n_total"].replace(0, 1)

print("=== Split ratio 예시 ===")
print(
    df_stats[
        ["dataset_name", "endpoint", "n_total", "train_ratio", "valid_ratio", "test_ratio"]
    ].sort_values("n_total", ascending=False).head(20).to_string(index=False)
)

=== Split ratio 예시 ===
 dataset_name            endpoint  n_total  train_ratio  valid_ratio  test_ratio
         ames                ames     7490     0.850734          0.0    0.149266
     tox21_df         tox21_NR-ER     7275     0.935808          0.0    0.064192
     tox21_df        tox21_NR-AhR     6051     0.933565          0.0    0.066435
     tox21_df        tox21_SR-MMP     6022     0.905015          0.0    0.094985
 herg_unified        herg_unified     5520     0.800362          0.0    0.199638
     tox21_df        tox21_SR-ARE     3867     0.849754          0.0    0.150246
     tox21_df     tox21_NR-ER-LBD     3588     0.918339          0.0    0.081661
   metabolism       cyp2c19_veith     2478     0.801049          0.0    0.198951
     tox21_df        tox21_SR-HSE     2458     0.923922          0.0    0.076078
   metabolism        cyp1a2_veith     2268     0.800705          0.0    0.199295
     tox21_df      tox21_SR-ATAD5     2115     0.937116          0.0    0.062884
     

/var/folders/r2/hf806xw17vv9p1hn3v10qr980000gn/T/ipykernel_6096/1165544243.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_stats["train_ratio"] = df_stats["n_train"] / df_stats["n_total"].replace(0, 1)
/var/folders/r2/hf806xw17vv9p1hn